# Bronze-Lite — Tipado y esquema canónico
**Pipeline stage:** `BRONZE` 

Bronze-lite aplica solo casteos de tipos y renombrado de columnas.
Sin transformaciones de negocio. La rawdata debe quedar reproducible bit-a-bit.

In [1]:
import os, json
import pandas as pd
import duckdb

# Leer el archivo raw más reciente
with open("../data/raw/latest.json") as f:
    latest = json.load(f)

RAW_PATH = latest["latest_raw"]
print(f"Cargando: {RAW_PATH}")
df = pd.read_csv(RAW_PATH)
print(f"Shape: {df.shape}")
df.dtypes

Cargando: ../data/raw/pscomppars_stellar_20260406T011715Z.csv
Shape: (6153, 18)


pl_name             object
hostname            object
sy_snum              int64
sy_pnum              int64
sy_dist            float64
st_spectype         object
st_teff            float64
st_rad             float64
st_mass            float64
st_lum             float64
st_logg            float64
st_met             float64
pl_orbper          float64
pl_rade            float64
pl_bmasse          float64
pl_eqt             float64
disc_year          float64
discoverymethod     object
dtype: object

## Esquema canónico — renombrado y tipos

In [2]:
# Renombrado canónico (snake_case explícito)
RENAME = {
    "pl_name"       : "pl_name",
    "hostname"      : "hostname",
    "sy_snum"       : "n_stars",
    "sy_pnum"       : "n_planets",
    "sy_dist"       : "dist_pc",
    "st_spectype"   : "st_spectype",
    "st_teff"       : "st_teff_k",
    "st_rad"        : "st_rad_rsun",
    "st_mass"       : "st_mass_msun",
    "st_lum"        : "st_lum_log",
    "st_logg"       : "st_logg",
    "st_met"        : "st_met_dex",
    "pl_orbper"     : "pl_orbper_d",
    "pl_rade"       : "pl_rad_rearth",
    "pl_bmasse"     : "pl_mass_mearth",
    "pl_eqt"        : "pl_eqt_k",
    "disc_year"     : "disc_year",
    "discoverymethod": "disc_method",
}

df_b = df.rename(columns=RENAME)[list(RENAME.values())].copy()

# Casteos de tipos
float_cols = [
    "dist_pc", "st_teff_k", "st_rad_rsun", "st_mass_msun",
    "st_lum_log", "st_logg", "st_met_dex",
    "pl_orbper_d", "pl_rad_rearth", "pl_mass_mearth", "pl_eqt_k",
]
int_cols = ["n_stars", "n_planets", "disc_year"]
str_cols = ["pl_name", "hostname", "st_spectype", "disc_method"]

for c in float_cols:
    df_b[c] = pd.to_numeric(df_b[c], errors="coerce")
for c in int_cols:
    df_b[c] = pd.to_numeric(df_b[c], errors="coerce").astype("Int64")
for c in str_cols:
    df_b[c] = df_b[c].astype(str).str.strip().replace("nan", pd.NA)

print("Tipado aplicado")
df_b.dtypes

Tipado aplicado


pl_name            object
hostname           object
n_stars             Int64
n_planets           Int64
dist_pc           float64
st_spectype        object
st_teff_k         float64
st_rad_rsun       float64
st_mass_msun      float64
st_lum_log        float64
st_logg           float64
st_met_dex        float64
pl_orbper_d       float64
pl_rad_rearth     float64
pl_mass_mearth    float64
pl_eqt_k          float64
disc_year           Int64
disc_method        object
dtype: object

## Guardar Bronze como Parquet

In [3]:
import pyarrow as pa, pyarrow.parquet as pq

os.makedirs("../data/bronze", exist_ok=True)
BRONZE_PATH = "../data/bronze/bronze_stellar.parquet"

table = pa.Table.from_pandas(df_b, preserve_index=False)
pq.write_table(table, BRONZE_PATH, compression="snappy")

size_kb = os.path.getsize(BRONZE_PATH) / 1024
size_csv_kb = os.path.getsize(RAW_PATH) / 1024
print(f"Parquet guardado : {BRONZE_PATH}")
print(f"Tamaño Parquet : {size_kb:,.0f} KB")
print(f"Tamaño CSV raw : {size_csv_kb:,.0f} KB")
print(f"Compresión     : {100*(1 - size_kb/size_csv_kb):.1f}% menor")

Parquet guardado : ../data/bronze/bronze_stellar.parquet
Tamaño Parquet : 380 KB
Tamaño CSV raw : 935 KB
Compresión     : 59.3% menor


## Validación de esquema con DuckDB

In [4]:
con = duckdb.connect()

result = con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{BRONZE_PATH}')
""").df()
print(result.to_string())

       column_name column_type null   key default extra
0          pl_name     VARCHAR  YES  None    None  None
1         hostname     VARCHAR  YES  None    None  None
2          n_stars      BIGINT  YES  None    None  None
3        n_planets      BIGINT  YES  None    None  None
4          dist_pc      DOUBLE  YES  None    None  None
5      st_spectype     VARCHAR  YES  None    None  None
6        st_teff_k      DOUBLE  YES  None    None  None
7      st_rad_rsun      DOUBLE  YES  None    None  None
8     st_mass_msun      DOUBLE  YES  None    None  None
9       st_lum_log      DOUBLE  YES  None    None  None
10         st_logg      DOUBLE  YES  None    None  None
11      st_met_dex      DOUBLE  YES  None    None  None
12     pl_orbper_d      DOUBLE  YES  None    None  None
13   pl_rad_rearth      DOUBLE  YES  None    None  None
14  pl_mass_mearth      DOUBLE  YES  None    None  None
15        pl_eqt_k      DOUBLE  YES  None    None  None
16       disc_year      BIGINT  YES  None    Non

## Perfil de nulos y duplicados

In [5]:
print("=== Nulos por columna ===")
null_pct = (df_b.isnull().sum() / len(df_b) * 100).round(1)
print(null_pct.to_string())

print(f"\n=== Duplicados en pl_name : {df_b['pl_name'].duplicated().sum()}")
print(f"=== Duplicados en hostname: {df_b.duplicated(subset=['hostname']).sum()} "
      f"(esperado > 0, hay estrellas con múltiples planetas)")

print(f"\n=== Estrellas únicas : {df_b['hostname'].nunique():,}")
print(f"=== Planetas únicos  : {df_b['pl_name'].nunique():,}")

=== Nulos por columna ===
pl_name            0.0
hostname           0.0
n_stars            0.0
n_planets          0.0
dist_pc            0.4
st_spectype       62.5
st_teff_k          4.7
st_rad_rsun        5.1
st_mass_msun       0.1
st_lum_log         5.0
st_logg            5.2
st_met_dex         8.9
pl_orbper_d        5.4
pl_rad_rearth      0.8
pl_mass_mearth     0.5
pl_eqt_k          25.4
disc_year          0.0
disc_method        0.0

=== Duplicados en pl_name : 0
=== Duplicados en hostname: 1568 (esperado > 0, hay estrellas con múltiples planetas)

=== Estrellas únicas : 4,585
=== Planetas únicos  : 6,153


## Checkpoint BRONZE
| Check | Estado |
|---|---|
| Columnas renombradas (snake_case) | ✓ |
| Tipos casteados (float/int/str) | ✓ |
| Guardado como Parquet + Snappy | ✓ |
| Perfil de nulos documentado | ✓ |
| Sin transformaciones de negocio | ✓ |